In [2]:
!nvidia-smi

Fri Jul 10 06:36:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 105.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.7 MB/s eta 0:00:00


In [4]:
import torch
import transformers
import peft
import bitsandbytes as bnb
import datasets
import trl

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("BitsAndBytes:", bnb.__version__)

print("\nCUDA available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())
print("GPU 0:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
Transformers: 5.13.0
PEFT: 0.19.1
BitsAndBytes: 0.49.2

CUDA available: True
Number of GPUs: 2
GPU 0: Tesla T4


In [5]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model
)

from datasets import Dataset

print("All required classes imported successfully")

All required classes imported successfully


In [6]:
!wget -q -O alpaca.zip \
"https://www.kaggle.com/api/v1/datasets/download/thedevastator/alpaca-language-instruction-training"

!unzip -q alpaca.zip -d alpaca_data

!find alpaca_data -type f

alpaca_data/train.csv


In [7]:
import pandas as pd

df = pd.read_csv("alpaca_data/train.csv")

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 3 rows:")
display(df.head(3))


Dataset shape: (51760, 3)

Column names:
['instruction', 'output', 'input']

First 3 rows:


,instruction,output,input
0,Give three tips for staying healthy.,1. Eat a balanced and nutritious diet: Make su...,NaN
1,What are the three primary colors?,"The three primary colors are red, blue, and ye...",NaN
2,Describe the structure of an atom.,An atom is the basic building block of all mat...,NaN


In [8]:
# Replace missing input values with empty text
df["input"] = df["input"].fillna("")

# Remove rows missing an instruction or output
df = df.dropna(
    subset=["instruction", "output"]
)

# Remove duplicate examples
df = df.drop_duplicates(
    subset=["instruction", "input", "output"]
).reset_index(drop=True)

# Shuffle reproducibly
df = df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Select 2,000 examples for free-GPU training
train_df = df.iloc[:2000].copy()

print("Cleaned dataset size:", len(df))
print("Selected training examples:", len(train_df))

print("\nMissing values:")
print(train_df.isnull().sum())

display(train_df.head(3))

Cleaned dataset size: 51756
Selected training examples: 2000

Missing values:
instruction    0
output         0
input          0
dtype: int64


,instruction,output,input
0,Determine whether the following statement is t...,The statement is mostly true. Most rocks are a...,
1,Give three tips for someone moving to a new city.,1. Research and Understand the Local Culture: ...,
2,Explain why renewable energy sources are impor...,Renewable energy sources are important because...,


In [9]:
train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

print(train_dataset)

print("\nFirst example:")
print(train_dataset[0])

Dataset({
    features: ['instruction', 'output', 'input'],
    num_rows: 2000
})

First example:
{'instruction': 'Determine whether the following statement is true or false: All rocks are made up of minerals.', 'output': 'The statement is mostly true. Most rocks are aggregates of minerals, formed through various geological processes. However, there are exceptions, such as coal, which forms from compressed plant remains, and obsidian, which is a type of volcanic glass, and these are not made up of minerals in the traditional sense.', 'input': ''}


In [10]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

# Ensure a padding token is available
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer loaded successfully")
print("Vocabulary size:", len(tokenizer))
print("EOS token:", tokenizer.eos_token)
print("PAD token:", tokenizer.pad_token)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded successfully
Vocabulary size: 151665
EOS token: <|im_end|>
PAD token: <|endoftext|>


In [11]:
def format_example(example):

    # Include additional input only when it exists
    if example["input"].strip():
        user_content = (
            f"{example['instruction']}\n\n"
            f"Additional input:\n{example['input']}"
        )
    else:
        user_content = example["instruction"]

    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": user_content
        },
        {
            "role": "assistant",
            "content": example["output"]
        }
    ]

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": formatted_text}


formatted_dataset = train_dataset.map(
    format_example
)

print(formatted_dataset)

print("\nFORMATTED EXAMPLE:\n")
print(formatted_dataset[0]["text"])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'output', 'input', 'text'],
    num_rows: 2000
})

FORMATTED EXAMPLE:

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Determine whether the following statement is true or false: All rocks are made up of minerals.<|im_end|>
<|im_start|>assistant
The statement is mostly true. Most rocks are aggregates of minerals, formed through various geological processes. However, there are exceptions, such as coal, which forms from compressed plant remains, and obsidian, which is a type of volcanic glass, and these are not made up of minerals in the traditional sense.<|im_end|>



In [12]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,

    # NF4 is designed for normally distributed neural-network weights
    bnb_4bit_quant_type="nf4",

    # Quantizes the quantization constants to save more memory
    bnb_4bit_use_double_quant=True,

    # T4 supports FP16 computation well
    bnb_4bit_compute_dtype=torch.float16
)

print(bnb_config)

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



In [13]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,

    # Use only the first T4 GPU
    device_map={"": 0},

    trust_remote_code=True,

    # Prevent accidental BF16 loading
    dtype=torch.float16
)

# Disable KV cache during training
model.config.use_cache = False

print("Model loaded successfully")
print("Model device:", model.device)
print("Loaded in 4-bit:", model.is_loaded_in_4bit)

allocated = torch.cuda.memory_allocated(0) / (1024 ** 3)
reserved = torch.cuda.memory_reserved(0) / (1024 ** 3)

print(f"GPU 0 memory allocated: {allocated:.2f} GB")
print(f"GPU 0 memory reserved: {reserved:.2f} GB")

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully
Model device: cuda:0
Loaded in 4-bit: True
GPU 0 memory allocated: 1.07 GB
GPU 0 memory reserved: 1.12 GB


In [14]:
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

print("Model prepared for k-bit training")

Model prepared for k-bit training


In [15]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],

    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(
    model,
    lora_config
)

# Keep all trainable LoRA parameters in FP32
# This prevents the BF16 gradient error from our previous attempt
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

model.print_trainable_parameters()

print("\nTrainable parameter data types:")

trainable_dtypes = {}

for name, param in model.named_parameters():
    if param.requires_grad:
        dtype = str(param.dtype)

        trainable_dtypes[dtype] = (
            trainable_dtypes.get(dtype, 0)
            + param.numel()
        )

print(trainable_dtypes)

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815

Trainable parameter data types:
{'torch.float32': 4358144}


In [16]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir="./qwen_qlora_output",

    # One complete pass through the 2,000 examples
    num_train_epochs=1,

    # 2 examples at a time
    per_device_train_batch_size=2,

    # Accumulate 4 batches before updating LoRA
    gradient_accumulation_steps=4,

    # Suitable learning rate for LoRA adapters
    learning_rate=2e-4,

    # Tesla T4 supports FP16
    fp16=True,
    bf16=False,

    # Memory-efficient optimizer
    optim="paged_adamw_8bit",

    # Print training information every 10 steps
    logging_steps=10,

    # Save after the epoch
    save_strategy="epoch",

    # Disable external logging
    report_to="none",

    # Dataset settings
    dataset_text_field="text",
    max_length=512,

    # Enable checkpointing through the trainer too
    gradient_checkpointing=True,

    remove_unused_columns=False,

    seed=42
)

print("SFT configuration created successfully")

SFT configuration created successfully


In [17]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=formatted_dataset,
    processing_class=tokenizer
)

print("SFTTrainer created successfully")

Adding EOS to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

SFTTrainer created successfully


In [19]:
# Convert all trainable parameters in the trainer model to FP32
for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

# Check their data types
trainable_dtypes = {}

for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        dtype = str(param.dtype)

        trainable_dtypes[dtype] = (
            trainable_dtypes.get(dtype, 0)
            + param.numel()
        )

print("Trainer model trainable dtypes:")
print(trainable_dtypes)

Trainer model trainable dtypes:
{'torch.float32': 4358144}


In [20]:
import time

torch.cuda.reset_peak_memory_stats(0)

start_time = time.time()

training_result = trainer.train()

training_time = time.time() - start_time

peak_memory = (
    torch.cuda.max_memory_allocated(0)
    / (1024 ** 3)
)

print("\nTraining completed successfully!")

print(
    f"Training time: "
    f"{training_time / 60:.2f} minutes"
)

print(
    f"Peak GPU memory allocated: "
    f"{peak_memory:.2f} GB"
)

print("\nTraining result:")
print(training_result)

Step,Training Loss
10,3.059869
20,2.609060
30,2.483289
40,2.349700
50,2.368699
60,2.309284
70,2.230891
80,2.282179
90,2.247087
100,2.234156



Training completed successfully!
Training time: 9.12 minutes
Peak GPU memory allocated: 2.88 GB

Training result:
TrainOutput(global_step=125, training_loss=2.395476577758789, metrics={'train_runtime': 546.7083, 'train_samples_per_second': 3.658, 'train_steps_per_second': 0.229, 'total_flos': 5144931578486784.0, 'train_loss': 2.395476577758789, 'entropy': 1.1670449197292327, 'num_tokens': 361897.0, 'mean_token_accuracy': 0.7020158439874649, 'epoch': 1.0})


In [21]:
adapter_path = "./qwen_qlora_adapter"

# Save only the trained LoRA adapter
trainer.model.save_pretrained(adapter_path)

# Save the tokenizer with it
tokenizer.save_pretrained(adapter_path)

print("LoRA adapter saved successfully!")
print("Saved at:", adapter_path)

LoRA adapter saved successfully!
Saved at: ./qwen_qlora_adapter


In [22]:
import gc

# Delete training objects and the 4-bit QLoRA model
del trainer
del model

# Clear Python and CUDA memory
gc.collect()
torch.cuda.empty_cache()

print("4-bit training model removed")

for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / (1024 ** 3)
    reserved = torch.cuda.memory_reserved(i) / (1024 ** 3)

    print(
        f"GPU {i} — allocated: {allocated:.2f} GB, "
        f"reserved: {reserved:.2f} GB"
    )

4-bit training model removed
GPU 0 — allocated: 1.56 GB, reserved: 2.04 GB
GPU 1 — allocated: 0.00 GB, reserved: 0.00 GB


In [23]:
from peft import PeftModel

# Load the original base model again WITHOUT 4-bit quantization
fp16_base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map={"": 0},
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

print("Fresh FP16 base model loaded successfully")
print("Model device:", fp16_base_model.device)
print("Model dtype:", fp16_base_model.dtype)

print(
    "Loaded in 4-bit:",
    getattr(fp16_base_model, "is_loaded_in_4bit", False)
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Fresh FP16 base model loaded successfully
Model device: cuda:0
Model dtype: torch.float16
Loaded in 4-bit: False


In [26]:
fp16_model_with_adapter = PeftModel.from_pretrained(
    fp16_base_model,
    adapter_path
)

print("LoRA adapter loaded into the FP16 base model")

fp16_model_with_adapter.print_trainable_parameters()

LoRA adapter loaded into the FP16 base model
trainable params: 0 || all params: 1,548,072,448 || trainable%: 0.0000


In [27]:
merged_model = fp16_model_with_adapter.merge_and_unload()

print("LoRA adapter merged successfully!")

print("Merged model type:")
print(type(merged_model))

print("\nModel dtype:")
print(merged_model.dtype)

LoRA adapter merged successfully!
Merged model type:
<class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>

Model dtype:
torch.float16


In [28]:
merged_model_path = "./qwen_qlora_merged_fp16"

merged_model.save_pretrained(
    merged_model_path,
    safe_serialization=True
)

tokenizer.save_pretrained(
    merged_model_path
)

print("Merged FP16 model saved successfully!")
print("Saved at:", merged_model_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged FP16 model saved successfully!
Saved at: ./qwen_qlora_merged_fp16


In [29]:
import gc

# Delete models currently stored on the GPU
del fp16_model_with_adapter
del fp16_base_model
del merged_model

# Clean Python and CUDA memory
gc.collect()

torch.cuda.empty_cache()

print("GPU model objects removed")

for i in range(torch.cuda.device_count()):

    allocated = (
        torch.cuda.memory_allocated(i)
        / (1024 ** 3)
    )

    reserved = (
        torch.cuda.memory_reserved(i)
        / (1024 ** 3)
    )

    print(
        f"GPU {i} — "
        f"allocated: {allocated:.2f} GB, "
        f"reserved: {reserved:.2f} GB"
    )

GPU model objects removed
GPU 0 — allocated: 3.76 GB, reserved: 4.44 GB
GPU 1 — allocated: 0.00 GB, reserved: 0.00 GB


In [30]:
from transformers import AutoModelForCausalLM, AutoTokenizer

cpu_tokenizer = AutoTokenizer.from_pretrained(
    merged_model_path
)

cpu_model = AutoModelForCausalLM.from_pretrained(
    merged_model_path,
    dtype=torch.float32,
    device_map="cpu",
    low_cpu_mem_usage=True
)

cpu_model.eval()

print("Merged model loaded successfully on CPU")
print("Model device:", next(cpu_model.parameters()).device)
print("Model dtype:", next(cpu_model.parameters()).dtype)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Merged model loaded successfully on CPU
Model device: cpu
Model dtype: torch.float32


In [31]:
import time

test_instruction = (
    "Give three practical tips for staying productive while studying."
)

messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": test_instruction
    }
]

# Apply Qwen's chat format
prompt = cpu_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Convert text into tensors and keep them on CPU
inputs = cpu_tokenizer(
    prompt,
    return_tensors="pt"
)

print("Input device:", inputs["input_ids"].device)

start_time = time.time()

with torch.inference_mode():

    outputs = cpu_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        repetition_penalty=1.1
    )

latency = time.time() - start_time

# Remove the input tokens and decode only the generated answer
generated_tokens = outputs[
    0,
    inputs["input_ids"].shape[1]:
]

response = cpu_tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("\nInstruction:")
print(test_instruction)

print("\nFine-tuned model response:")
print(response)

print(
    f"\nCPU inference latency: "
    f"{latency:.2f} seconds"
)

print(
    f"Generated tokens: "
    f"{len(generated_tokens)}"
)

print(
    f"Generation speed: "
    f"{len(generated_tokens) / latency:.2f} tokens/second"
)

Input device: cpu

Instruction:
Give three practical tips for staying productive while studying.

Fine-tuned model response:
1. Create a schedule: Make sure to set aside specific times each day or week for studying and stick to it as closely as possible. This will help you stay on track and avoid procrastination.

2. Eliminate distractions: Find a quiet, comfortable place to study where you won't be interrupted by phone calls, social media, or other distractions. Turn off notifications on your devices and close unnecessary tabs in your browser.

3. Take breaks: It's important to take regular breaks during long periods of studying to prevent burnout and maintain focus. Try taking short walks, stretching exercises, or meditating to recharge your mind and body.

CPU inference latency: 37.03 seconds
Generated tokens: 129
Generation speed: 3.48 tokens/second


In [32]:
import gc

# Remove the fine-tuned CPU model from RAM
del cpu_model

gc.collect()

print("Fine-tuned CPU model removed from RAM")

Fine-tuned CPU model removed from RAM


In [33]:
base_cpu_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32,
    device_map="cpu",
    low_cpu_mem_usage=True
)

base_cpu_model.eval()

print("Original base model loaded on CPU")
print("Model device:", next(base_cpu_model.parameters()).device)
print("Model dtype:", next(base_cpu_model.parameters()).dtype)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Original base model loaded on CPU
Model device: cpu
Model dtype: torch.float32


In [34]:
import time

# Use the exact same prompt as before
base_inputs = cpu_tokenizer(
    prompt,
    return_tensors="pt"
)

start_time = time.time()

with torch.inference_mode():

    base_outputs = base_cpu_model.generate(
        **base_inputs,
        max_new_tokens=150,
        do_sample=False,
        repetition_penalty=1.1
    )

base_latency = time.time() - start_time

# Decode only newly generated tokens
base_generated_tokens = base_outputs[
    0,
    base_inputs["input_ids"].shape[1]:
]

base_response = cpu_tokenizer.decode(
    base_generated_tokens,
    skip_special_tokens=True
)

print("Instruction:")
print(test_instruction)

print("\nOriginal base model response:")
print(base_response)

print(
    f"\nBase-model CPU latency: "
    f"{base_latency:.2f} seconds"
)

print(
    f"Generated tokens: "
    f"{len(base_generated_tokens)}"
)

print(
    f"Generation speed: "
    f"{len(base_generated_tokens) / base_latency:.2f} tokens/second"
)

Instruction:
Give three practical tips for staying productive while studying.

Original base model response:
1. Create a schedule: Make sure to set aside specific times each day or week for studying and stick to it as closely as possible. This will help you stay on track and avoid procrastination.

2. Eliminate distractions: Find a quiet place to study where you won't be disturbed by phone calls, social media, or other interruptions. Turn off your phone or put it in another room so that you can focus on your work without being distracted.

3. Take breaks: It's important to take regular breaks during long periods of studying to prevent burnout and maintain productivity. Try taking short walks, doing some stretching exercises, or listening to music to keep yourself energized and focused.

Base-model CPU latency: 40.00 seconds
Generated tokens: 139
Generation speed: 3.48 tokens/second


In [36]:
import pandas as pd

comparison_results = pd.DataFrame({
    "Metric": [
        "Model",
        "Device",
        "Inference Precision",
        "CPU Latency",
        "Generated Tokens",
        "Generation Speed",
        "Output Quality"
    ],

    "Original Base Model": [
        "Qwen2.5-1.5B-Instruct",
        "CPU",
        "FP32",
        f"{base_latency:.2f} seconds",
        len(base_generated_tokens),
        (
            f"{len(base_generated_tokens) / base_latency:.2f} "
            "tokens/second"
        ),
        "Clear, relevant, and well structured"
    ],

    "QLoRA Fine-Tuned Model": [
        "Qwen2.5-1.5B-Instruct + merged LoRA",
        "CPU",
        "FP32",
        f"{latency:.2f} seconds",
        len(generated_tokens),
        (
            f"{len(generated_tokens) / latency:.2f} "
            "tokens/second"
        ),
        "Clear, relevant, well structured, and slightly more concise"
    ]
})

display(comparison_results)

,Metric,Original Base Model,QLoRA Fine-Tuned Model
0,Model,Qwen2.5-1.5B-Instruct,Qwen2.5-1.5B-Instruct + merged LoRA
1,Device,CPU,CPU
2,Inference Precision,FP32,FP32
3,CPU Latency,40.00 seconds,37.03 seconds
4,Generated Tokens,139,129
5,Generation Speed,3.48 tokens/second,3.48 tokens/second
6,Output Quality,"Clear, relevant, and well structured","Clear, relevant, well structured, and slightly..."


In [37]:
print("=" * 80)
print("QLoRA FINE-TUNING AND CPU INFERENCE — FINAL RESULTS")
print("=" * 80)

print("\nTRAINING RESULTS")
print("-" * 80)

print("Model: Qwen2.5-1.5B-Instruct")
print("Training method: QLoRA")
print("Quantization: 4-bit NF4")
print("Double quantization: Enabled")
print("Compute dtype: FP16")

print("\nLoRA configuration:")
print("Rank: 16")
print("Alpha: 32")
print("Dropout: 0.05")
print("Target modules: q_proj, k_proj, v_proj, o_proj")

print("\nTraining configuration:")
print("Epochs: 1")
print("Batch size per GPU: 2")
print("Gradient accumulation steps: 4")
print("Learning rate: 2e-4")

print("\nTrainable parameters:")
print("4,358,144 / 1,548,072,448")
print("Trainable percentage: 0.2815%")

print("\nTraining performance:")
print("Training time: 9.12 minutes")
print("Peak GPU memory: 2.88 GB")
print("Final training loss: 2.3955")
print("Mean token accuracy: 70.20%")

print("\n" + "=" * 80)
print("CPU INFERENCE COMPARISON")
print("=" * 80)

print(
    f"\nOriginal model latency: "
    f"{base_latency:.2f} seconds"
)

print(
    f"Fine-tuned model latency: "
    f"{latency:.2f} seconds"
)

print(
    "\nOriginal generation speed: "
    f"{len(base_generated_tokens) / base_latency:.2f} "
    "tokens/second"
)

print(
    "Fine-tuned generation speed: "
    f"{len(generated_tokens) / latency:.2f} "
    "tokens/second"
)

print("\nObservation:")
print(
    "Both models generated coherent and relevant responses. "
    "The QLoRA fine-tuned model produced a slightly more concise "
    "response while maintaining the requested three-point structure. "
    "Inference speed remained nearly identical because LoRA was merged "
    "into the same 1.5B model architecture."
)

print("\n" + "=" * 80)
print("WORKFLOW COMPLETED SUCCESSFULLY")
print("=" * 80)

QLoRA FINE-TUNING AND CPU INFERENCE — FINAL RESULTS

TRAINING RESULTS
--------------------------------------------------------------------------------
Model: Qwen2.5-1.5B-Instruct
Training method: QLoRA
Quantization: 4-bit NF4
Double quantization: Enabled
Compute dtype: FP16

LoRA configuration:
Rank: 16
Alpha: 32
Dropout: 0.05
Target modules: q_proj, k_proj, v_proj, o_proj

Training configuration:
Epochs: 1
Batch size per GPU: 2
Gradient accumulation steps: 4
Learning rate: 2e-4

Trainable parameters:
4,358,144 / 1,548,072,448
Trainable percentage: 0.2815%

Training performance:
Training time: 9.12 minutes
Peak GPU memory: 2.88 GB
Final training loss: 2.3955
Mean token accuracy: 70.20%

CPU INFERENCE COMPARISON

Original model latency: 40.00 seconds
Fine-tuned model latency: 37.03 seconds

Original generation speed: 3.48 tokens/second
Fine-tuned generation speed: 3.48 tokens/second

Observation:
Both models generated coherent and relevant responses. The QLoRA fine-tuned model produced